# stock_process_dl1: Stock Market Next-Day Prediction

This notebook provides a full end-to-end stock prediction workflow focused on next-day direction and a simple iterative 30-day directional forecast.

## What this notebook covers
1. Data collection from Yahoo Finance
2. Feature engineering (moving averages, return, Bollinger Bands)
3. Target construction for next-day movement
4. Model training with RandomForestClassifier
5. Evaluation using precision, recall, and classification report
6. Next 30 days directional projection

## Conceptual overview
Stock market prediction is a time-series problem. A practical pipeline usually includes:

- Data Collection: OHLCV prices and related market context
- Feature Engineering: transforming raw prices into predictive signals
- Model Selection: choosing models suitable for noisy sequential data
- Training Loop: prediction, loss, optimization (for gradient-based models)
- Evaluation: measuring quality with fit-for-purpose metrics

In [ ]:
import pandas as pd
import yfinance as yf
from datetime import date, timedelta
from sklearn.metrics import classification_report, precision_score, recall_score
from sklearn.ensemble import RandomForestClassifier

## Configuration
Adjust these values to change symbol, date range, and forecast horizon.

In [ ]:
ticker = "AAPL"
start_date = "2015-01-01"
end_date = date.today()
prediction_date = (date.today() + timedelta(days=1)).strftime("%Y-%m-%d")
predictors = ["Close", "High", "Low", "Open", "Volume", "20_MA", "50_MA", "Daily_Return", "Upper_Band", "Lower_Band"]

## Forecast helper
This class creates an iterative directional forecast for the next N days using the trained classifier.

In [ ]:
class NextDaysPredictions:
    def __init__(self, data, model, predictors, next_days):
        self.data = data
        self.model = model
        self.predictors = predictors
        self.next_days = next_days

    def build(self, stock_data, predictors, next_days):
        last_row = stock_data.iloc[-1].copy()
        next_days_predictions = []

        for _ in range(next_days):
            next_day_data = last_row[predictors].values.reshape(1, -1)
            next_day_prediction = self.model.predict(next_day_data)[0]
            next_days_predictions.append(next_day_prediction)

            # Keep feature vector stable for a simple directional rollout
            last_row[predictors] = next_day_data[0]

        next_days_df = pd.DataFrame({
            "Date": pd.date_range(start=stock_data["Date"].iloc[-1] + pd.Timedelta(days=1), periods=next_days),
            "Prediction": next_days_predictions
        })
        return next_days_df

## Feature engineering
Builds target and indicator columns used by the model.

In [ ]:
def init_target_indicators(stock_data):
    close_prices = stock_data["Close"]
    if isinstance(close_prices, pd.DataFrame):
        close_prices = close_prices.iloc[:, 0]

    stock_data["Next_Close"] = stock_data["Close"].shift(-1)
    stock_data["20_MA"] = stock_data["Close"].rolling(window=20).mean()
    stock_data["50_MA"] = stock_data["Close"].rolling(window=50).mean()
    stock_data["Daily_Return"] = stock_data["Close"].pct_change()
    stock_data["20_STD"] = stock_data["Close"].rolling(window=20).std()
    stock_data["Upper_Band"] = stock_data["20_MA"] + (2 * stock_data["20_STD"])
    stock_data["Lower_Band"] = stock_data["20_MA"] - (2 * stock_data["20_STD"])
    return stock_data.copy()

## Download and prepare data
Downloads market data, applies indicators, and materializes a clean feature table.

In [ ]:
sp500 = yf.download("^GSPC", start=start_date, end=end_date)

stock_data = yf.download(ticker, start=start_date, end=end_date)
stock_data = init_target_indicators(stock_data)
stock_data.to_csv("stock_data.csv", index=True)

stock_data_n = stock_data.reset_index()
if isinstance(stock_data_n.columns, pd.MultiIndex):
    stock_data_n.columns = stock_data_n.columns.droplevel(1)

stock_data_n = stock_data_n[[
    "Date", "Close", "High", "Low", "Open", "Volume",
    "Next_Close", "20_MA", "50_MA", "Daily_Return",
    "Upper_Band", "Lower_Band"
]].copy()

prd_stock_data = stock_data_n.copy()
stock_data_n.fillna(stock_data_n.mean(numeric_only=True), inplace=True)
stock_data_n.to_csv("prd_stock_data.csv", index=True)
stock_data_n.head()

## Target variable
Target1 is 1 when next close is greater than current close, else 0.

In [ ]:
close_prices = prd_stock_data["Close"]
if isinstance(close_prices, pd.DataFrame):
    close_prices = close_prices.iloc[:, 0]

prd_stock_data["Target1"] = (prd_stock_data["Next_Close"] > close_prices).astype(int)
prd_stock_data.head()

## Train RandomForest model
This model predicts next-day direction from engineered features.

In [ ]:
train_data = prd_stock_data[:180]
test_data = prd_stock_data[30:]

model = RandomForestClassifier(min_samples_split=15, n_estimators=600, random_state=1)
model.fit(prd_stock_data[predictors], prd_stock_data["Target1"])

predictions = model.predict(test_data[predictors])
preds = pd.Series(predictions, index=test_data.index)
print("predictions (first 10):", preds.head(10).to_list())

## Evaluate model
Precision and recall summarize directional quality; the full report includes class-level details.

In [ ]:
precision = precision_score(test_data["Target1"], predictions)
recall = recall_score(test_data["Target1"], predictions)
print(f"Precision: {precision:.2f}")
print(f"Recall:    {recall:.2f}")
print()
print(classification_report(test_data["Target1"], predictions))

## Next 30 days forecast
Generates a simple directional projection table with up or down labels.

In [ ]:
next_days_df = NextDaysPredictions(
    data=prd_stock_data, model=model, predictors=predictors, next_days=30
).build(stock_data=prd_stock_data, predictors=predictors, next_days=30)

next_days_df["Trend"] = next_days_df["Prediction"].map({1: "up", 0: "down"})
next_days_df